In [1]:
import re
import io
import pandas as pd
from os import listdir
from os.path import isfile, join

In [3]:
path = "./Papers"
files = [f for f in listdir(path) if isfile(join(path, f))]

dfs = []
for file in files:
    sub_df = pd.read_excel(f"./Papers/{file}", skiprows=[0])
    dfs.append(sub_df)

df = pd.concat(dfs, ignore_index=True)
df.reset_index(inplace=True, drop=True)
ids = ["DOI", "PMID", "PMCID", "ISBN"]

In [ ]:
# # Testing
# df = pd.read_excel("Benchmark-03-09.xlsx")
# ids = ["DOI"]

In [4]:
# Print Shape
print(f"Shape: {df.shape}")

# Print Columns
print("Columns")
for column in df.columns:
    print(f"\t{column}")

Shape: (36038, 55)
Columns
	Rank
	Publication ID
	DOI
	PMID
	PMCID
	ISBN
	Title
	Abstract
	Acknowledgements
	Funding
	Source title
	Anthology title
	Book editors
	Publisher
	ISSN
	MeSH terms
	Publication date
	PubYear
	Publication date (online)
	Publication date (print)
	Volume
	Issue
	Pagination
	Open Access
	Publication Type
	Document Type
	Authors
	Authors (Raw Affiliation)
	Corresponding Authors
	Authors Affiliations
	Research Organizations - standardized
	GRID IDs
	City of standardized research organization
	State of standardized research organization
	Country of standardized research organization
	Funder
	Funder Group
	Funder Country
	Grant IDs of Supporting Grants
	Supporting Grants
	Times cited
	Recent citations
	RCR
	FCR
	Altmetric
	Source Linkout
	Dimensions URL
	Fields of Research (ANZSRC 2020)
	RCDC Categories
	HRCS HC Categories
	HRCS RAC Categories
	Cancer Types
	CSO Categories
	Units of Assessment
	Sustainable Development Goals


In [5]:
# Drop Rows with Missing Values
df.dropna(subset=["Title", "Abstract"], inplace=True, ignore_index=True)
print(f"Shape: {df.shape}")

df.dropna(subset=ids, inplace=True, ignore_index=True, how="all")
print(f"Shape: {df.shape}")

Shape: (36038, 55)
Shape: (36038, 55)


In [6]:
# Drop Duplicates
df.drop_duplicates(subset=["Abstract"], inplace=True, ignore_index=True)
print(f"Shape: {df.shape}")

for paper_id in ids:
    df.drop_duplicates(subset=[paper_id, "Title"], inplace=True, ignore_index=True)
    print(f"Shape (Dropped Rows with Duplicate {paper_id} and Title): {df.shape}")

Shape: (35943, 55)
Shape (Dropped Rows with Duplicate DOI and Title): (35943, 55)
Shape (Dropped Rows with Duplicate PMID and Title): (35874, 55)
Shape (Dropped Rows with Duplicate PMCID and Title): (35833, 55)
Shape (Dropped Rows with Duplicate ISBN and Title): (35820, 55)


In [7]:
# Fields of Research to Exclude
# Any row that has any of these fields 
# will be excluded.
fields_of_research_to_exclude = [
    "3101 Biochemistry and Cell Biology", 
    "3105 Genetics", 
    "34 Chemical Sciences", 
    "3401 Analytical Chemistry", 
    "3215 Reproductive Medicine",
    "3215 Reproductive Medicine",
    "33 Built Environment and Design",
    "34 Chemical Sciences",
    "3401 Analytical Chemistry",
    "3402 Inorganic Chemistry",
    "3403 Macromolecular and Materials Chemistry",
    "3404 Medicinal and Biomolecular Chemistry",
    "3406 Physical Chemistry",
    "40 Engineering",
    "4001 Aerospace Engineering",
    "4003 Biomedical Engineering",
    "4004 Chemical Engineering",
    "4016 Materials Engineering",
    "3402 Inorganic Chemistry",
    "3703 Geochemistry",
    "32 Biomedical and Clinical Sciences",
    "3207 Medical Microbiology",
    "3106 Industrial Biotechnology",
]

In [8]:
# Store Counts of Fields of Research
fields_of_research = df["Fields of Research (ANZSRC 2020)"].tolist()
field_counts = {}

for field_of_research in fields_of_research:
    fields = re.split(r";\s?", field_of_research)
    # Update Count
    for field in fields:
        field_counts[field] = field_counts.get(field, 0) + 1

# Sort Counts
field_counts_items = [*field_counts.items()]
field_counts_items.sort(key=lambda field_count: field_count[1], reverse=True)

# Print Counts
print("Fields of Research in Data")
for field, count in field_counts_items:
    print(f"{field}: {count}")

Fields of Research in Data
31 Biological Sciences: 32249
3103 Ecology: 24482
41 Environmental Sciences: 12250
3109 Zoology: 7929
30 Agricultural, Veterinary and Food Sciences: 3944
4102 Ecological Applications: 3163
3108 Plant Biology: 2914
3107 Microbiology: 2845
4104 Environmental Management: 2366
4101 Climate Change Impacts and Adaptation: 1926
32 Biomedical and Clinical Sciences: 1081
4105 Pollution and Contamination: 1062
3007 Forestry Sciences: 954
4103 Environmental Biotechnology: 851
3004 Crop and Pasture Production: 838
3101 Biochemistry and Cell Biology: 710
3104 Evolutionary Biology: 693
37 Earth Sciences: 638
3105 Genetics: 546
4106 Soil Sciences: 458
3005 Fisheries Sciences: 423
40 Engineering: 326
34 Chemical Sciences: 308
3003 Animal Production: 271
3207 Medical Microbiology: 264
3106 Industrial Biotechnology: 233
3708 Oceanography: 180
3009 Veterinary Sciences: 174
3202 Clinical Sciences: 147
52 Psychology: 142
3006 Food Sciences: 132
4011 Environmental Engineering: 116

In [9]:
# Remove Rows by Fields of Research
df = df[~df["Fields of Research (ANZSRC 2020)"].str.contains(rf"{'|'.join(fields_of_research_to_exclude)}")]
df.reset_index(inplace=True, drop=True)
print(f"Shape: {df.shape}")

Shape: (33020, 55)


In [10]:
# Save Cleaned Data
df.to_csv("CleanedPapers.csv", index=False, encoding="utf-8")